Model making

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def conv_block(x, filters, dilation=1, name=None):
    x = layers.Conv2D(filters, 3, padding='same', dilation_rate=dilation, name=f'{name}_conv' if name else None)(x)
    x = layers.BatchNormalization(name=f'{name}_bn' if name else None)(x)
    x = layers.ReLU(name=f'{name}_relu' if name else None)(x)
    return x

def MSCFF_V2(input_shape=(512, 512, 3), n_classes=2):
    inputs = layers.Input(shape=input_shape, name='imageinput')

    # Encoder
    x1 = conv_block(inputs, 64, name='block1_1')
    x1 = conv_block(x1, 64, name='block1_2')
    p1 = layers.MaxPooling2D(2, name='pool1')(x1)
    
    x2 = conv_block(p1, 128, name='block2_1')
    x2 = conv_block(x2, 128, name='block2_2')
    p2 = layers.MaxPooling2D(2, name='pool2')(x2)
    
    x3 = conv_block(p2, 256, name='block3_1')
    x3 = conv_block(x3, 256, name='block3_2')
    p3 = layers.MaxPooling2D(2, name='pool3')(x3)
    
    x4 = conv_block(p3, 512, name='block4_1')
    x4 = conv_block(x4, 512, name='block4_2')

    # Multi-scale dilated convolutions
    d1 = conv_block(x4, 512, dilation=2, name='dilate2')
    d2 = conv_block(x4, 512, dilation=5, name='dilate5')
    d3 = conv_block(x4, 512, dilation=3, name='dilate3')
    d4 = conv_block(x4, 512, dilation=5, name='dilate5_2')
    features = layers.Add(name='msc_add')([x4, d1, d2, d3, d4])

    # Decoder
    u3 = layers.Conv2DTranspose(256, 4, strides=2, padding='same', name='up3')(features)
    u3 = layers.Concatenate()([u3, x3])
    d3 = conv_block(u3, 256, name='dec3')

    u2 = layers.Conv2DTranspose(128, 4, strides=2, padding='same', name='up2')(d3)
    u2 = layers.Concatenate()([u2, x2])
    d2 = conv_block(u2, 128, name='dec2')

    u1 = layers.Conv2DTranspose(64, 4, strides=2, padding='same', name='up1')(d2)
    u1 = layers.Concatenate()([u1, x1])
    d1 = conv_block(u1, 64, name='dec1')

    outputs = layers.Conv2D(n_classes, 1, activation='softmax', name='out_conv')(d1)

    return models.Model(inputs, outputs, name='MSCFF_V2')

# Example model instantiation:
model = MSCFF_V2(input_shape=(512, 512, 3), n_classes=2)
model.summary()

Data pipeline

In [ ]:
import tensorflow as tf
import os

def parse_image(img_path: str, mask_path: str, img_size=(512, 512)):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, img_size)
    img = tf.cast(img, tf.float32) / 255.0

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, img_size, method='nearest')
    mask = tf.cast(mask, tf.uint8)
    mask = tf.squeeze(mask, axis=-1)
    return img, mask

def get_dataset(img_dir, mask_dir, batch_size=1, img_size=(512,512)):
    img_paths = sorted([os.path.join(img_dir, fname) for fname in os.listdir(img_dir)])
    mask_paths = sorted([os.path.join(mask_dir, fname) for fname in os.listdir(mask_dir)])
    ds = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))
    ds = ds.map(lambda x, y: parse_image(x, y, img_size), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

Training setup

In [ ]:
import os

train_img_dir = '/path/to/train/images'
train_mask_dir = '/path/to/train/masks'
val_img_dir = '/path/to/val/images'
val_mask_dir = '/path/to/val/masks'

train_ds = get_dataset(train_img_dir, train_mask_dir, batch_size=1)
val_ds = get_dataset(val_img_dir, val_mask_dir, batch_size=1)

model = MSCFF_V2(input_shape=(1024, 1024, 3))
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
              loss='categorical_crossentropy',
              metrics=[
                    'accuracy',
                    tf.keras.metrics.OneHotMeanIoU(num_classes=3, name='mean_iou'),
                    tf.keras.metrics.F1Score(name='F-1_score'),
                    tf.keras.metrics.Precision(name='precision'),
                    tf.keras.metrics.Recall(name='recall')])

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_mean_iou'),
    tf.keras.callbacks.ModelCheckpoint('checkpoint/mscff_v2-{epoch:02d}.h5', save_best_only=True, monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
]

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)

Save and load model file

In [ ]:
# After training
model.save('mscff_v2_model')

# ... some time later, maybe in a new script:
from tensorflow import keras
model = keras.models.load_model('mscff_v2_model')

# Use model for prediction
pred = model.predict(test_images)